# 01. Khám phá dữ liệu — ViMMSD EDA

Notebook phân tích toàn diện bộ dữ liệu **ViMMSD** với 4 nhãn:
- `not-sarcasm` — không mỉa mai
- `text-sarcasm` — mỉa mai chủ yếu qua văn bản
- `image-sarcasm` — mỉa mai chủ yếu qua hình ảnh
- `multi-sarcasm` — mỉa mai qua cả văn bản lẫn hình ảnh

**Mục tiêu:** phân bố nhãn & imbalance · đặc trưng caption · emoji/teencode/hashtag · phân tích từ ngữ · đặc trưng ảnh · sample gallery · tiền xử lý · tổng kết định hướng

## 0. Bootstrap — clone repo & cài thư viện

In [ ]:
# === Cell bootstrap (giống nhau ở mọi notebook, xem CODEBASE.md mục 6) ===
import os, subprocess, sys
from pathlib import Path

REPO = "github.com/yunaLee21/Multimodal_Sarcasm_Detection.git"
REF = "main"               # branch của mình, hoặc 1 commit hash để tái lặp kết quả
CODE_DIR = "/tmp/IE403"    # clone ra ngoài /kaggle/working để token không bị lưu vào output


def get_github_token():
    """Token cho repo private; repo public thì không cần tạo secret."""
    try:
        if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
            from kaggle_secrets import UserSecretsClient  # type: ignore
            return UserSecretsClient().get_secret("GITHUB_TOKEN")
        from google.colab import userdata  # type: ignore
        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return None


ON_CLOUD = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or "COLAB_RELEASE_TAG" in os.environ
if ON_CLOUD:
    if "COLAB_RELEASE_TAG" in os.environ:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
    token = get_github_token()
    url = f"https://{token}@{REPO}" if token else f"https://{REPO}"
    subprocess.run(["rm", "-rf", CODE_DIR])
    subprocess.run(["git", "clone", "-q", url, CODE_DIR], check=True)
    subprocess.run(["git", "-C", CODE_DIR, "checkout", "-q", REF], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    f"{CODE_DIR}/requirements-kaggle.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", CODE_DIR], check=True)
    if CODE_DIR + "/src" not in sys.path:
        sys.path.insert(0, CODE_DIR + "/src")
    os.chdir(CODE_DIR)
elif Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

print("cwd:", os.getcwd())
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("code commit:", commit or "unknown")

## 1. Import & Load dữ liệu

In [ ]:
import json
import re
import logging
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
from PIL import Image
from IPython.display import display, HTML

from vimmsd.utils.config import load_config
from vimmsd.data.dataset import load_all_records
from vimmsd.data.preprocessing import TEENCODE, WORD_RE

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.WARNING)

# ── Style chung ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor": "#ffffff",
    "axes.facecolor":   "#ffffff",
    "axes.edgecolor":   "#e5e7eb",
    "axes.labelcolor":  "#111827",
    "xtick.color":      "#374151",
    "ytick.color":      "#374151",
    "text.color":       "#111827",
    "grid.color":       "#f3f4f6",
    "grid.linestyle":   "--",
    "axes.grid":        True,
    "font.family":      "DejaVu Sans",
    "savefig.facecolor": "#ffffff",
    "figure.dpi":       120,
})

LABEL_COLORS = {
    "not-sarcasm":    "#0284c7",
    "text-sarcasm":   "#ea580c",
    "image-sarcasm":  "#9333ea",
    "multi-sarcasm":  "#16a34a",
}
print("✅ Import xong.")

In [ ]:
cfg = load_config("configs/base.yaml")
LABELS = list(cfg.data.labels)
COLORS = [LABEL_COLORS[l] for l in LABELS]

splits_raw = load_all_records(cfg)

rows = []
for split_name, recs in splits_raw.items():
    for r in recs:
        rows.append({**r, "split": split_name})
df = pd.DataFrame(rows)

id2label = {i: l for i, l in enumerate(LABELS)}
df["label_name"] = df["label"].map(lambda i: id2label.get(i, "(unlabeled)"))
labeled = df[df["label"] >= 0].copy()

print(f"Tổng mẫu: {len(df):,}  |  Có nhãn: {len(labeled):,}  |  Không nhãn: {len(df)-len(labeled):,}")
print("\nSố mẫu theo split:")
display(df.groupby("split").size().rename("count").to_frame())

## 2. Phân bố nhãn

In [ ]:
for split in ["train", "val", "test"]:
    sub = labeled[labeled["split"] == split]
    counts = sub["label_name"].value_counts().reindex(LABELS, fill_value=0)
    pct = (counts / counts.sum() * 100).round(2)
    df_stat = pd.DataFrame({"Số mẫu": counts, "Tỉ lệ (%)": pct})
    df_stat.loc["TỔNG"] = df_stat.sum()
    df_stat.loc["TỔNG", "Tỉ lệ (%)"] = 100.0
    print(f"\n──── Split: {split.upper()} ({len(sub):,} mẫu) ────")
    display(df_stat)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Phân bố nhãn theo split", fontsize=15, fontweight="bold", color="#111827", y=1.02)

for ax, split in zip(axes, ["train", "val", "test"]):
    sub = labeled[labeled["split"] == split]
    counts = sub["label_name"].value_counts().reindex(LABELS, fill_value=0)
    bars = ax.bar(LABELS, counts.values, color=COLORS, edgecolor="#d1d5db", linewidth=0.8, zorder=3)
    ax.set_title(f"{split} ({len(sub):,} mẫu)", fontsize=12, fontweight="bold")
    ax.set_xlabel("Nhãn", fontsize=10)
    ax.set_ylabel("Số mẫu", fontsize=10)
    ax.set_xticks(range(len(LABELS)))
    ax.set_xticklabels(LABELS, rotation=15, ha="right", fontsize=9)
    ax.yaxis.grid(True, zorder=0)
    ax.set_axisbelow(True)
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, str(val),
                ha="center", va="bottom", fontsize=9, color="#111827", fontweight="bold")

plt.tight_layout()
Path("reports/figures").mkdir(parents=True, exist_ok=True)
plt.savefig("reports/figures/eda_01_label_distribution.png", dpi=150,
            bbox_inches="tight", facecolor="#ffffff")
plt.show()
print("💾 Đã lưu: reports/figures/eda_01_label_distribution.png")

In [ ]:
train_labeled = labeled[labeled["split"] == "train"]
counts_train = train_labeled["label_name"].value_counts().reindex(LABELS, fill_value=0)
max_count = counts_train.max()

print("\n📊 Imbalance ratio (so với lớp đông nhất):")
df_imb = pd.DataFrame({
    "Số mẫu": counts_train,
    "IR (max/lớp)": (max_count / counts_train).round(2),
})
display(df_imb)
ir_max = (max_count / counts_train.min()).round(2)
print(f"\n⚠️  IR tổng: {ir_max}× — '{counts_train.idxmax()}' nhiều hơn '{counts_train.idxmin()}' {ir_max}× lần")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
split_names = ["train", "val", "test"]
bottom = np.zeros(len(split_names))

for i, label in enumerate(LABELS):
    vals = []
    for sp in split_names:
        sub = labeled[labeled["split"] == sp]
        total = len(sub)
        cnt = (sub["label_name"] == label).sum()
        vals.append(cnt / total * 100 if total else 0)
    vals = np.array(vals)
    ax.bar(split_names, vals, bottom=bottom, color=LABEL_COLORS[label],
           label=label, edgecolor="#d1d5db", linewidth=0.5)
    for j, (b, v) in enumerate(zip(bottom, vals)):
        if v > 3:
            ax.text(j, b + v/2, f"{v:.1f}%", ha="center", va="center",
                    fontsize=8, color="#111827", fontweight="bold")
    bottom += vals

ax.set_ylabel("Tỉ lệ (%)", fontsize=11)
ax.set_title("Tỉ lệ nhãn theo split (stratified)", fontsize=13, fontweight="bold")
ax.legend(loc="upper right", framealpha=0.3, fontsize=9)
ax.set_ylim(0, 110)
plt.tight_layout()
plt.savefig("reports/figures/eda_02_label_stacked.png", dpi=150,
            bbox_inches="tight", facecolor="#ffffff")
plt.show()

## 3. Phân tích Caption (Text)

In [ ]:
labeled = labeled.copy()
labeled["n_chars"]  = labeled["caption"].str.len()
labeled["n_words"]  = labeled["caption"].str.split().str.len()
labeled["n_tokens"] = labeled["caption"].apply(lambda t: len(WORD_RE.findall(t)))

print("📝 Thống kê độ dài caption theo nhãn (n_words):")
display(
    labeled.groupby("label_name")["n_words"]
    .describe(percentiles=[.25, .5, .75, .95])
    .reindex(LABELS)
    .round(1)
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Phân phối độ dài caption theo nhãn", fontsize=13, fontweight="bold")

for ax, metric, title in zip(axes, ["n_words", "n_chars"], ["Số từ", "Số ký tự"]):
    data_by_label = [labeled[labeled["label_name"] == l][metric].values for l in LABELS]
    parts = ax.violinplot(data_by_label, positions=range(len(LABELS)),
                          showmedians=True, showextrema=True)
    for i, (pc, col) in enumerate(zip(parts["bodies"], COLORS)):
        pc.set_facecolor(col)
        pc.set_alpha(0.7)
        pc.set_edgecolor(col)
    parts["cmedians"].set_color("#111827")
    parts["cmedians"].set_linewidth(2)
    for key in ["cmaxes", "cmins", "cbars"]:
        parts[key].set_color("#6b7280")
    ax.set_xticks(range(len(LABELS)))
    ax.set_xticklabels(LABELS, rotation=12, ha="right", fontsize=9)
    ax.set_ylabel(title, fontsize=10)
    ax.set_title(f"{title} theo nhãn", fontsize=11)

plt.tight_layout()
plt.savefig("reports/figures/eda_03_caption_length.png", dpi=150,
            bbox_inches="tight", facecolor="#ffffff")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
bins = np.arange(0, labeled["n_words"].quantile(0.99) + 5, 2)

for label, color in zip(LABELS, COLORS):
    subset = labeled[labeled["label_name"] == label]["n_words"]
    ax.hist(subset, bins=bins, alpha=0.55, label=label, color=color, edgecolor="none")

ax.set_xlabel("Số từ trong caption", fontsize=11)
ax.set_ylabel("Số mẫu", fontsize=11)
ax.set_title("Phân phối số từ caption (99th pct)", fontsize=12, fontweight="bold")
ax.legend(fontsize=9, framealpha=0.3)
plt.tight_layout()
plt.savefig("reports/figures/eda_04_caption_hist.png", dpi=150,
            bbox_inches="tight", facecolor="#ffffff")
plt.show()

print(f"\nGợi ý max_length cho tokenizer:")
for q, label_q in [(0.90, "P90"), (0.95, "P95"), (0.99, "P99")]:
    v_w = labeled["n_words"].quantile(q)
    v_c = labeled["n_chars"].quantile(q)
    v_t = labeled["n_tokens"].quantile(q)
    print(f"  {label_q}: n_words={v_w:.0f}  n_chars={v_c:.0f}  n_tokens={v_t:.0f}")

## 4. Emoji, Hashtag & Teencode

In [ ]:
import emoji as emoji_lib

labeled["n_emoji"]      = labeled["caption"].map(emoji_lib.emoji_count)
labeled["has_hashtag"]  = labeled["caption"].str.contains(r"#\w", regex=True)
labeled["n_teencode"]   = labeled["caption"].map(
    lambda t: sum(w.lower() in TEENCODE for w in WORD_RE.findall(t))
)
labeled["has_url"]       = labeled["caption"].str.contains(r"https?://|www\.", regex=True)
labeled["has_mention"]   = labeled["caption"].str.contains(r"@\w", regex=True)
labeled["empty_caption"] = labeled["caption"].str.strip() == ""

FEAT_COLS = ["n_emoji", "has_hashtag", "n_teencode", "has_url", "has_mention", "empty_caption"]
print("📊 Trung bình đặc trưng văn bản theo nhãn:")
display(
    labeled.groupby("label_name")[FEAT_COLS].mean().reindex(LABELS).round(4)
)

In [ ]:
feat_means = labeled.groupby("label_name")[FEAT_COLS].mean().reindex(LABELS)

fig, ax = plt.subplots(figsize=(10, 4))
cmap = LinearSegmentedColormap.from_list("eda", ["#f8fafc", "#c084fc", "#9333ea"])
im = ax.imshow(feat_means.values.T, aspect="auto", cmap=cmap)

ax.set_xticks(range(len(LABELS)))
ax.set_xticklabels(LABELS, rotation=12, ha="right", fontsize=10)
ax.set_yticks(range(len(FEAT_COLS)))
ax.set_yticklabels(FEAT_COLS, fontsize=10)
ax.set_title("Heatmap đặc trưng văn bản (trung bình) theo nhãn",
             fontsize=12, fontweight="bold")

for i in range(len(FEAT_COLS)):
    for j in range(len(LABELS)):
        val = feat_means.values[j, i]
        ax.text(j, i, f"{val:.3f}", ha="center", va="center", fontsize=9,
                color="#111827" if val < feat_means.values.max()*0.6 else "black")

plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.savefig("reports/figures/eda_05_text_feature_heatmap.png", dpi=150,
            bbox_inches="tight", facecolor="#ffffff")
plt.show()

In [ ]:
all_emojis = Counter(
    e["emoji"] for t in labeled["caption"]
    for e in emoji_lib.emoji_list(t)
)
top_emojis = all_emojis.most_common(20)

fig, ax = plt.subplots(figsize=(12, 4))
if top_emojis:
    emojis, emoji_counts = zip(*top_emojis)
    ax.bar(range(len(emojis)), emoji_counts, color="#9333ea",
           edgecolor="#d1d5db", linewidth=0.5)
    ax.set_xticks(range(len(emojis)))
    ax.set_xticklabels(emojis, fontsize=14)
    for i, cnt in enumerate(emoji_counts):
        ax.text(i, cnt + 1, str(cnt), ha="center", va="bottom",
                fontsize=8, color="#111827")
ax.set_ylabel("Tần suất", fontsize=10)
ax.set_title("Top-20 Emoji phổ biến nhất (toàn bộ labeled)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("reports/figures/eda_06_top_emoji.png", dpi=150,
            bbox_inches="tight", facecolor="#ffffff")
plt.show()
print("\nTop-20 emoji:", top_emojis)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 8))
fig.suptitle("Top-10 Emoji theo từng nhãn", fontsize=13, fontweight="bold")

for ax, label, color in zip(axes.flat, LABELS, COLORS):
    sub = labeled[labeled["label_name"] == label]
    ec = Counter(e["emoji"] for t in sub["caption"] for e in emoji_lib.emoji_list(t))
    top = ec.most_common(10)
    if not top:
        ax.text(0.5, 0.5, "Không có emoji", ha="center", va="center",
                transform=ax.transAxes)
        ax.set_title(label, color=color, fontweight="bold")
        continue
    emj, cnts = zip(*top)
    ax.bar(range(len(emj)), cnts, color=color, edgecolor="#d1d5db", alpha=0.85)
    ax.set_xticks(range(len(emj)))
    ax.set_xticklabels(emj, fontsize=13)
    ax.set_title(f"{label}  (n={len(sub):,})", color=color,
                 fontsize=11, fontweight="bold")
    ax.set_ylabel("Tần suất", fontsize=9)

plt.tight_layout()
plt.savefig("reports/figures/eda_07_emoji_by_label.png", dpi=150,
            bbox_inches="tight", facecolor="#ffffff")
plt.show()

In [ ]:
print("📊 Các teencode xuất hiện nhiều nhất theo nhãn:")
for label in LABELS:
    sub = labeled[labeled["label_name"] == label]
    tc_cnt = Counter(
        w.lower()
        for t in sub["caption"]
        for w in WORD_RE.findall(t)
        if w.lower() in TEENCODE
    )
    top10 = tc_cnt.most_common(10)
    print(f"  [{label}]: {top10}")

## 5. Phân tích từ ngữ đặc trưng (Word Frequency)

In [ ]:
import string

STOPWORDS_VI = {
    "và", "của", "là", "có", "được", "cho", "với", "một", "trong", "này",
    "đó", "thì", "mà", "hay", "cũng", "đã", "đang", "sẽ", "rất", "lại",
    "nên", "khi", "ra", "vào", "lên", "theo", "bị", "như", "về", "đến",
    "từ", "nhưng", "vì", "nếu", "thế", "tôi", "em", "anh", "chị", "bạn",
    "mình", "tui", "ông", "bà", "con", "các", "những", "để", "không",
    "hơn", "vẫn", "còn", "đây", "họ", "chúng", "ta", "at", "https",
}


def top_words(series, n=30, stopwords=STOPWORDS_VI):
    counter = Counter()
    for text in series:
        words = WORD_RE.findall(text.lower())
        words = [w for w in words if len(w) > 1 and w not in stopwords
                 and not w.isnumeric() and w not in string.punctuation]
        counter.update(words)
    return counter.most_common(n)


top_by_label = {label: top_words(labeled[labeled["label_name"] == label]["caption"])
                for label in LABELS}

print("Top-15 từ (sau khi loại stopword) theo nhãn:")
for label in LABELS:
    top = top_by_label[label][:15]
    print(f"  [{label}]: {[f'{w}({c})' for w, c in top]}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Top-15 từ xuất hiện nhiều nhất theo nhãn",
             fontsize=13, fontweight="bold")

for ax, label, color in zip(axes.flat, LABELS, COLORS):
    top = top_by_label[label][:15]
    if not top:
        ax.set_visible(False)
        continue
    words, cnts = zip(*top)
    y_pos = range(len(words))
    bars = ax.barh(list(y_pos), list(cnts), color=color,
                   alpha=0.8, edgecolor="#d1d5db")
    ax.set_yticks(list(y_pos))
    ax.set_yticklabels(list(words), fontsize=10)
    ax.invert_yaxis()
    ax.set_xlabel("Tần suất", fontsize=9)
    n_label = len(labeled[labeled["label_name"] == label])
    ax.set_title(f"{label}  (n={n_label:,})",
                 color=color, fontsize=11, fontweight="bold")
    for bar, cnt in zip(bars, cnts):
        ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                str(cnt), va="center", fontsize=8, color="#111827")

plt.tight_layout()
plt.savefig("reports/figures/eda_08_top_words_by_label.png", dpi=150,
            bbox_inches="tight", facecolor="#ffffff")
plt.show()

## 6. Phân tích Hình ảnh

In [ ]:
from tqdm.auto import tqdm

img_meta = []
broken = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Đọc metadata ảnh"):
    try:
        with Image.open(row["image_path"]) as im:
            img_meta.append({
                "id":           row["id"],
                "split":        row["split"],
                "label_name":   row["label_name"],
                "label":        row["label"],
                "width":        im.size[0],
                "height":       im.size[1],
                "format":       im.format or "unknown",
                "mode":         im.mode,
                "n_pixels":     im.size[0] * im.size[1],
                "aspect_ratio": im.size[0] / im.size[1],
            })
    except Exception:
        broken.append(row["image_path"])

df_img = pd.DataFrame(img_meta)
print(f"\n✅ Đọc được {len(df_img):,} ảnh | Lỗi: {len(broken)} ảnh")
if broken:
    print("Ảnh lỗi:", broken[:5])

In [ ]:
print("📐 Thống kê kích thước ảnh (toàn bộ dataset):")
display(df_img[["width", "height", "n_pixels", "aspect_ratio"]].describe().round(1))

print("\n📋 Phân bố format ảnh:")
display(df_img["format"].value_counts())

print("\n🎨 Phân bố color mode:")
display(df_img["mode"].value_counts())

In [ ]:
labeled_img = df_img[df_img["label"] >= 0].copy()

fig, ax = plt.subplots(figsize=(9, 7))
for label, color in zip(LABELS, COLORS):
    sub = labeled_img[labeled_img["label_name"] == label]
    sub_s = sub.sample(min(500, len(sub)), random_state=42)
    ax.scatter(sub_s["width"], sub_s["height"], alpha=0.35, s=15,
               color=color, label=label)

xlim = ax.get_xlim()
x_range = np.linspace(0, xlim[1], 100)
for ratio, ls, lc, lbl in [
    (1,    "--", "#aaa", "AR=1 (vuông)"),
    (4/3,  ":",  "#888", "AR=4/3"),
    (16/9, "-.", "#777", "AR=16/9"),
]:
    ax.plot(x_range, x_range/ratio, linestyle=ls, color=lc, linewidth=1, label=lbl)

ax.set_xlabel("Chiều rộng (px)", fontsize=11)
ax.set_ylabel("Chiều cao (px)", fontsize=11)
ax.set_title("Phân bố kích thước ảnh theo nhãn", fontsize=12, fontweight="bold")
ax.legend(fontsize=8, framealpha=0.3, ncol=2)
plt.tight_layout()
plt.savefig("reports/figures/eda_09_image_size_scatter.png", dpi=150,
            bbox_inches="tight", facecolor="#ffffff")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
bins_ar = np.linspace(0.2, 3.0, 40)
for label, color in zip(LABELS, COLORS):
    sub = labeled_img[labeled_img["label_name"] == label]["aspect_ratio"]
    ax.hist(sub, bins=bins_ar, alpha=0.5, color=color, label=label, edgecolor="none")
ax.axvline(1.0, color="#111827", linestyle="--", linewidth=1, label="AR=1 (vuông)")
ax.set_xlabel("Aspect Ratio (w/h)", fontsize=10)
ax.set_ylabel("Số ảnh", fontsize=10)
ax.set_title("Phân phối aspect ratio theo nhãn", fontsize=11, fontweight="bold")
ax.legend(fontsize=8, framealpha=0.3)

ax2 = axes[1]
data_px = [labeled_img[labeled_img["label_name"] == l]["n_pixels"].values / 1e6
           for l in LABELS]
bp = ax2.boxplot(data_px, patch_artist=True, showfliers=False)
for patch, color in zip(bp["boxes"], COLORS):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
for elem in ["whiskers", "caps", "medians"]:
    for line in bp[elem]:
        line.set_color("#4b5563")
ax2.set_xticks(range(1, len(LABELS)+1))
ax2.set_xticklabels(LABELS, rotation=12, ha="right", fontsize=9)
ax2.set_ylabel("Triệu pixels (MP)", fontsize=10)
ax2.set_title("Kích thước ảnh (Megapixels) theo nhãn",
              fontsize=11, fontweight="bold")

plt.tight_layout()
plt.savefig("reports/figures/eda_10_image_stats.png", dpi=150,
            bbox_inches="tight", facecolor="#ffffff")
plt.show()

## 7. Mẫu minh hoạ theo từng nhãn (Gallery)

In [ ]:
valid_paths = set(df_img["id"].tolist())
labeled_valid = labeled[labeled["id"].isin(valid_paths)].copy()
N_SAMPLES = 5

fig = plt.figure(figsize=(18, 4.5 * len(LABELS)))
fig.patch.set_facecolor("#ffffff")
outer = gridspec.GridSpec(len(LABELS), 1, hspace=0.4, figure=fig)

for row_idx, (label, color) in enumerate(zip(LABELS, COLORS)):
    inner = gridspec.GridSpecFromSubplotSpec(
        1, N_SAMPLES, subplot_spec=outer[row_idx], wspace=0.05)
    pool = labeled_valid[labeled_valid["label_name"] == label]
    subset = pool.sample(min(N_SAMPLES, len(pool)), random_state=7)

    for col_idx, (_, row_data) in enumerate(subset.iterrows()):
        ax = fig.add_subplot(inner[col_idx])
        try:
            img = Image.open(row_data["image_path"]).convert("RGB")
            ax.imshow(img)
        except Exception:
            ax.set_facecolor("#f1f5f9")
        ax.axis("off")
        cap = row_data["caption"]
        cap_short = cap[:50] + "…" if len(cap) > 50 else cap
        ax.set_title(cap_short, fontsize=7, color="#374151", pad=3, loc="left")
        if col_idx == 0:
            ax.text(-0.05, 0.5, label, transform=ax.transAxes,
                    va="center", ha="right", fontsize=11, fontweight="bold",
                    color=color, rotation=90)

fig.suptitle("Mẫu minh hoạ theo từng nhãn (5 mẫu / nhãn)",
             fontsize=14, fontweight="bold", color="#111827", y=1.005)
plt.savefig("reports/figures/eda_11_sample_gallery.png", dpi=120,
            bbox_inches="tight", facecolor="#ffffff")
plt.show()
print("💾 Đã lưu: reports/figures/eda_11_sample_gallery.png")

## 8. Phân tích đặc biệt: Caption ngắn/dài theo nhãn

In [ ]:
SHORT_THRESHOLD = 3
labeled["is_short"] = labeled["n_words"] < SHORT_THRESHOLD

print(f"Caption rỗng hoặc < {SHORT_THRESHOLD} từ theo nhãn:")
display(
    labeled.groupby("label_name")[["empty_caption", "is_short"]].sum().reindex(LABELS)
)

# text-sarcasm ngắn
ts_short = labeled[
    (labeled["label_name"] == "text-sarcasm") & (labeled["n_words"] < SHORT_THRESHOLD)
][["id", "caption", "n_words"]].head(10)
print(f"\ntext-sarcasm với caption < {SHORT_THRESHOLD} từ:")
display(ts_short)

# image-sarcasm caption dài
p75_global = labeled["n_words"].quantile(0.75)
is_long = labeled[
    (labeled["label_name"] == "image-sarcasm") & (labeled["n_words"] > p75_global)
]
pct_long = len(is_long) / max(1, (labeled["label_name"]=="image-sarcasm").sum()) * 100
print(f"\nimage-sarcasm với caption dài (> P75={p75_global:.0f} từ): "
      f"{len(is_long)} mẫu ({pct_long:.1f}%)")
display(is_long[["id", "caption", "n_words"]].sample(
    min(5, len(is_long)), random_state=0))

## 9. Tiền xử lý: Trước vs Sau

In [ ]:
from vimmsd.data.preprocessing import TextPreprocessor

pre = TextPreprocessor.from_config(cfg.data.text)

print("=" * 80)
print("Demo tiền xử lý caption")
print("=" * 80)
for label in LABELS:
    pool = labeled[labeled["label_name"] == label]
    samples = pool.sample(min(2, len(pool)), random_state=99)
    print(f"\n[{label}]")
    for _, row_data in samples.iterrows():
        raw = row_data["caption"]
        processed = pre(raw)
        print(f"  GỐC  : {raw[:120]}")
        print(f"  SAU  : {processed[:120]}")
        print()

In [ ]:
SAMPLE_SIZE = min(200, len(labeled))
sample_df = labeled.sample(SAMPLE_SIZE, random_state=0).copy()
sample_df["caption_clean"] = sample_df["caption"].map(pre)
sample_df["n_words_clean"] = sample_df["caption_clean"].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].scatter(sample_df["n_words"], sample_df["n_words_clean"],
                alpha=0.4, s=20, color="#9333ea")
lim = max(sample_df[["n_words", "n_words_clean"]].max())
axes[0].plot([0, lim], [0, lim], "--", color="#f59e0b", linewidth=1)
axes[0].set_xlabel("Số từ gốc", fontsize=10)
axes[0].set_ylabel("Số từ sau tiền xử lý", fontsize=10)
axes[0].set_title("Trước vs Sau tiền xử lý (n từ)", fontsize=11, fontweight="bold")

delta = sample_df["n_words_clean"] - sample_df["n_words"]
axes[1].hist(delta, bins=30, color="#0284c7", edgecolor="#d1d5db", alpha=0.8)
axes[1].axvline(0, color="#f59e0b", linestyle="--", linewidth=1.5)
axes[1].set_xlabel("Thay đổi số từ (sau - trước)", fontsize=10)
axes[1].set_ylabel("Số mẫu", fontsize=10)
axes[1].set_title("Phân phối thay đổi số từ", fontsize=11, fontweight="bold")
axes[1].text(0.02, 0.95, f"Mean Δ = {delta.mean():.2f}",
             transform=axes[1].transAxes, fontsize=9, color="#111827", va="top")

plt.tight_layout()
plt.savefig("reports/figures/eda_12_preprocessing_effect.png", dpi=150,
            bbox_inches="tight", facecolor="#ffffff")
plt.show()

## 10. Tổng kết & Định hướng xử lý

In [ ]:
import datetime

print("=" * 70)
print("TỔNG KẾT EDA — ViMMSD")
print("=" * 70)

counts_all = labeled["label_name"].value_counts().reindex(LABELS, fill_value=0)
total_labeled = len(labeled)
print(f"\n[1] Tổng mẫu có nhãn: {total_labeled:,}")
print(f"    Mẫu public test (không nhãn): {len(df) - total_labeled:,}")
print()
for label in LABELS:
    cnt = counts_all[label]
    pct = cnt / total_labeled * 100
    print(f"    {label:<20}: {cnt:>5,}  ({pct:.1f}%)")

ir_total = (max_count / counts_train.min()).round(2)
print(f"\n    ⚠️  Imbalance Ratio: {ir_total}×  "
      f"({counts_train.idxmax()} vs {counts_train.idxmin()})")

print(f"\n[2] Caption:")
print(f"    Trung vị số từ:  {labeled['n_words'].median():.0f} từ")
print(f"    P95 số từ:       {labeled['n_words'].quantile(0.95):.0f} từ")
print(f"    Có emoji:        {(labeled['n_emoji']>0).mean()*100:.1f}% mẫu")
print(f"    Có hashtag:      {labeled['has_hashtag'].mean()*100:.1f}% mẫu")
print(f"    Có teencode:     {(labeled['n_teencode']>0).mean()*100:.1f}% mẫu")
print(f"    Caption rỗng:   {labeled['empty_caption'].sum()} mẫu")

if len(df_img) > 0:
    print(f"\n[3] Hình ảnh:")
    print(f"    Ảnh đọc được: {len(df_img):,} / {len(df):,}")
    print(f"    Ảnh lỗi:      {len(broken)} ảnh")
    print(f"    Format phổ biến: {df_img['format'].value_counts().index[0]}")
    print(f"    Median (W×H): {df_img['width'].median():.0f}×{df_img['height'].median():.0f} px")
    print(f"    Median AR:    {df_img['aspect_ratio'].median():.2f}")

print()
print("=" * 70)
print("ĐỊNH HƯỚNG XỬ LÝ")
print("=" * 70)
print("""
A. MẤT CÂN BẰNG LỚP
   - text-sarcasm có tỉ lệ thấp nhất → cần weighted loss hoặc oversampling
   - Thử: class_weight=inverse, FocalLoss (gamma=2), WeightedRandomSampler

B. VĂN BẢN (Caption)
   - Nhiều emoji & teencode → giữ normalize_teencode=true, emoji=demojize
   - max_length nên ≥ 128 token (P95 từ, nhân thêm ~1.2x cho word_segment)
   - Caption rỗng: thay bằng chuỗi đặc biệt [EMPTY]
   - word_segment (underthesea) tăng chất lượng embedding PhoBERT

C. HÌNH ẢNH
   - Ảnh lỗi nhỏ (<1%) → thay bằng ảnh đen (dataset.py đã xử lý)
   - Aspect ratio đa dạng → resize về 224×224 (CLIP xử lý sẵn)
   - image-sarcasm cần encoder đủ mạnh (CLIP/ViT)

D. CHIẾN LƯỢC FUSION
   - multi-sarcasm cần cả 2 modality → cross-attention fusion ưu tiên
   - text-sarcasm: chú trọng text branch
   - image-sarcasm: chú trọng image branch
   - Ablation: text-only → image-only → concat → cross-attention
""")

# Lưu summary
summary_path = Path("reports/eda_summary.md")
summary_path.parent.mkdir(parents=True, exist_ok=True)
lines = [
    f"# EDA Summary — ViMMSD\n",
    f"_Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}_\n\n",
    "## Phân bố nhãn (toàn bộ labeled)\n",
    "| Nhãn | Số mẫu | Tỉ lệ |\n",
    "|------|--------|-------|\n",
]
for label in LABELS:
    cnt = counts_all[label]
    pct = cnt / total_labeled * 100
    lines.append(f"| {label} | {cnt:,} | {pct:.1f}% |\n")
lines += [
    f"\n**Imbalance Ratio:** {ir_total}×\n\n",
    "## Đặc trưng caption\n",
    f"- Trung vị n_words: {labeled['n_words'].median():.0f}\n",
    f"- P95 n_words: {labeled['n_words'].quantile(0.95):.0f}\n",
    f"- Tỉ lệ có emoji: {(labeled['n_emoji']>0).mean()*100:.1f}%\n",
    f"- Tỉ lệ có hashtag: {labeled['has_hashtag'].mean()*100:.1f}%\n",
    f"- Tỉ lệ có teencode: {(labeled['n_teencode']>0).mean()*100:.1f}%\n",
]
summary_path.write_text("".join(lines), encoding="utf-8")
print(f"\n💾 Đã lưu tóm tắt EDA: {summary_path}")

figures = sorted(Path("reports/figures").glob("eda_*.png"))
print("\n✅ EDA hoàn tất! Figures đã lưu:")
for f in figures:
    print(f"  📊 {f.name}")